# Task 1: Environment Setup, Data Loading & EDA
## Real-Time Face Segmentation for Movie Cast Identification

In this notebook we:
1. Install and import dependencies
2. Load the `.npy` dataset
3. Understand the data structure (images + bounding-box annotations)
4. Convert bounding-box annotations to binary face masks
5. Visualize samples and perform exploratory data analysis

## 1.1 Install Dependencies

In [ ]:
!pip install -q tensorflow==2.15.0 opencv-python==4.9.0.80 numpy==1.26.4 pandas==2.2.1 matplotlib==3.8.3 seaborn==0.13.2 scikit-learn==1.4.1.post1

## 1.2 Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns
import cv2
import os
from collections import Counter

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
print('All libraries imported successfully!')

## 1.3 Load the Dataset

In [ ]:
# Load the .npy file
data = np.load('Part 1- Train data - images.npy', allow_pickle=True)

print(f'Dataset shape: {data.shape}')
print(f'Total samples: {len(data)}')
print(f'Each sample contains: [image (numpy array), annotations (list of dicts)]')

## 1.4 Understand the Data Structure

Each sample is a pair: `[image, annotations]`
- **image**: A numpy array (H, W, 3) — the movie scene screenshot
- **annotations**: A list of dicts, each representing one face bounding box with keys:
  - `label`: Always `['Face']`
  - `points`: 2 points (top-left, bottom-right) in normalized coordinates (0-1)
  - `imageWidth`, `imageHeight`: Original image dimensions

In [ ]:
# Inspect a single sample
sample_img = data[0][0]
sample_annotations = data[0][1]

print('--- Image ---')
print(f'  Shape: {sample_img.shape}')
print(f'  Dtype: {sample_img.dtype}')
print(f'  Pixel range: [{sample_img.min()}, {sample_img.max()}]')

print('\n--- Annotations ---')
print(f'  Number of faces: {len(sample_annotations)}')
for j, ann in enumerate(sample_annotations):
    print(f'  Face {j+1}:')
    print(f'    Label: {ann["label"]}')
    print(f'    Bounding box points (normalized): {ann["points"]}')
    print(f'    Image size: {ann["imageWidth"]}x{ann["imageHeight"]}')

## 1.5 Extract Dataset Statistics

In [ ]:
# Collect statistics across all samples
image_heights = []
image_widths = []
num_channels = []
face_counts = []
face_areas_ratio = []  # face area as fraction of image area

for i in range(len(data)):
    img = data[i][0]
    anns = data[i][1]
    
    h, w = img.shape[0], img.shape[1]
    c = img.shape[2] if len(img.shape) == 3 else 1
    
    image_heights.append(h)
    image_widths.append(w)
    num_channels.append(c)
    face_counts.append(len(anns))
    
    for ann in anns:
        pts = ann['points']
        x1, y1 = pts[0]['x'], pts[0]['y']
        x2, y2 = pts[1]['x'], pts[1]['y']
        face_w = abs(x2 - x1)
        face_h = abs(y2 - y1)
        face_areas_ratio.append(face_w * face_h)

# Summary DataFrame
stats_df = pd.DataFrame({
    'Height': image_heights,
    'Width': image_widths,
    'Channels': num_channels,
    'Num_Faces': face_counts
})

print('=== Dataset Summary ===')
print(f'Total images: {len(data)}')
print(f'Total face annotations: {sum(face_counts)}')
print(f'\nImage dimensions:')
print(stats_df[['Height', 'Width']].describe().round(1))
print(f'\nChannel distribution: {Counter(num_channels)}')
print(f'  Note: 1=grayscale, 3=RGB, 4=RGBA')
print(f'\nFaces per image:')
print(stats_df['Num_Faces'].describe().round(2))

## 1.6 Data Visualizations

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Distribution of faces per image
ax = axes[0, 0]
face_count_series = pd.Series(face_counts)
face_count_series.value_counts().sort_index().plot(kind='bar', ax=ax, color='steelblue', edgecolor='black')
ax.set_title('Distribution of Faces per Image')
ax.set_xlabel('Number of Faces')
ax.set_ylabel('Count')

# 2. Image height distribution
ax = axes[0, 1]
ax.hist(image_heights, bins=30, color='coral', edgecolor='black', alpha=0.7)
ax.set_title('Image Height Distribution')
ax.set_xlabel('Height (pixels)')
ax.set_ylabel('Count')

# 3. Image width distribution
ax = axes[1, 0]
ax.hist(image_widths, bins=30, color='mediumseagreen', edgecolor='black', alpha=0.7)
ax.set_title('Image Width Distribution')
ax.set_xlabel('Width (pixels)')
ax.set_ylabel('Count')

# 4. Face area ratio distribution
ax = axes[1, 1]
ax.hist(face_areas_ratio, bins=40, color='mediumpurple', edgecolor='black', alpha=0.7)
ax.set_title('Face Area Ratio Distribution (fraction of image)')
ax.set_xlabel('Face Area / Image Area')
ax.set_ylabel('Count')

plt.tight_layout()
plt.savefig('eda_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: eda_distributions.png')

In [ ]:
# Scatter plot: Image dimensions
fig, ax = plt.subplots(figsize=(10, 6))
scatter = ax.scatter(image_widths, image_heights, c=face_counts, cmap='viridis', 
                     alpha=0.6, edgecolors='black', linewidth=0.5, s=50)
plt.colorbar(scatter, label='Number of Faces')
ax.set_xlabel('Width (pixels)')
ax.set_ylabel('Height (pixels)')
ax.set_title('Image Dimensions (colored by face count)')
ax.axhline(y=256, color='red', linestyle='--', alpha=0.5, label='Target 256px')
ax.axvline(x=256, color='red', linestyle='--', alpha=0.5)
ax.legend()
plt.tight_layout()
plt.savefig('eda_dimensions_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

## 1.7 Visualize Sample Images with Bounding Boxes

In [ ]:
def draw_bboxes(img, annotations):
    """Draw face bounding boxes on an image."""
    img_display = img.copy()
    h, w = img.shape[0], img.shape[1]
    
    # Convert grayscale/RGBA to RGB for display
    if len(img_display.shape) == 2:
        img_display = cv2.cvtColor(img_display, cv2.COLOR_GRAY2RGB)
    elif img_display.shape[2] == 4:
        img_display = cv2.cvtColor(img_display, cv2.COLOR_RGBA2RGB)
    
    for ann in annotations:
        pts = ann['points']
        x1 = int(pts[0]['x'] * w)
        y1 = int(pts[0]['y'] * h)
        x2 = int(pts[1]['x'] * w)
        y2 = int(pts[1]['y'] * h)
        cv2.rectangle(img_display, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(img_display, 'Face', (x1, y1 - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)
    
    return img_display


# Show 8 random samples with bounding boxes
np.random.seed(42)
sample_indices = np.random.choice(len(data), 8, replace=False)

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
for idx, ax in zip(sample_indices, axes.flatten()):
    img = data[idx][0]
    anns = data[idx][1]
    img_with_boxes = draw_bboxes(img, anns)
    ax.imshow(img_with_boxes)
    ax.set_title(f'Sample {idx} | {len(anns)} face(s) | {img.shape[1]}x{img.shape[0]}')
    ax.axis('off')

plt.suptitle('Sample Images with Face Bounding Boxes', fontsize=16, y=1.02)
plt.tight_layout()
plt.savefig('eda_sample_images.png', dpi=150, bbox_inches='tight')
plt.show()

## 1.8 Generate Binary Face Masks from Annotations

Since the dataset provides bounding-box annotations (not pixel-level masks), we create binary masks by filling the bounding box regions. This is our ground truth for the U-Net segmentation model.

In [ ]:
def create_binary_mask(img_shape, annotations):
    """Create a binary mask from bounding box annotations.
    
    Args:
        img_shape: (H, W) or (H, W, C) shape of the image
        annotations: list of annotation dicts with 'points' key
    
    Returns:
        Binary mask of shape (H, W) with 1s inside face regions
    """
    h, w = img_shape[0], img_shape[1]
    mask = np.zeros((h, w), dtype=np.uint8)
    
    for ann in annotations:
        pts = ann['points']
        x1 = int(pts[0]['x'] * w)
        y1 = int(pts[0]['y'] * h)
        x2 = int(pts[1]['x'] * w)
        y2 = int(pts[1]['y'] * h)
        # Ensure correct ordering
        x1, x2 = min(x1, x2), max(x1, x2)
        y1, y2 = min(y1, y2), max(y1, y2)
        mask[y1:y2, x1:x2] = 1
    
    return mask


# Generate masks for all images
print('Generating binary masks for all images...')
all_masks = []
for i in range(len(data)):
    img = data[i][0]
    anns = data[i][1]
    mask = create_binary_mask(img.shape, anns)
    all_masks.append(mask)

print(f'Generated {len(all_masks)} masks')
print(f'Sample mask shape: {all_masks[0].shape}')
print(f'Sample mask unique values: {np.unique(all_masks[0])}')

In [ ]:
# Visualize images alongside their generated masks
fig, axes = plt.subplots(3, 4, figsize=(20, 15))
sample_indices_mask = np.random.choice(len(data), 6, replace=False)

for i, idx in enumerate(sample_indices_mask):
    img = data[idx][0]
    mask = all_masks[idx]
    
    # Handle different channel formats
    if len(img.shape) == 2:
        img_rgb = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
    elif img.shape[2] == 4:
        img_rgb = cv2.cvtColor(img, cv2.COLOR_RGBA2RGB)
    else:
        img_rgb = img
    
    row = i // 2
    col = (i % 2) * 2
    
    # Original image
    axes[row, col].imshow(img_rgb)
    axes[row, col].set_title(f'Image {idx}')
    axes[row, col].axis('off')
    
    # Binary mask
    axes[row, col + 1].imshow(mask, cmap='gray')
    axes[row, col + 1].set_title(f'Mask {idx} | {len(data[idx][1])} face(s)')
    axes[row, col + 1].axis('off')

plt.suptitle('Images and Their Binary Face Masks', fontsize=16, y=1.02)
plt.tight_layout()
plt.savefig('eda_masks_visualization.png', dpi=150, bbox_inches='tight')
plt.show()

## 1.9 Mask Coverage Analysis

In [ ]:
# Analyze how much of each image is covered by face masks
mask_coverage = []
for mask in all_masks:
    coverage = mask.sum() / mask.size
    mask_coverage.append(coverage)

mask_coverage = np.array(mask_coverage)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Coverage histogram
axes[0].hist(mask_coverage * 100, bins=30, color='steelblue', edgecolor='black', alpha=0.7)
axes[0].set_title('Face Mask Coverage Distribution')
axes[0].set_xlabel('Coverage (%)')
axes[0].set_ylabel('Count')
axes[0].axvline(x=mask_coverage.mean() * 100, color='red', linestyle='--', label=f'Mean: {mask_coverage.mean()*100:.1f}%')
axes[0].legend()

# Coverage vs face count
axes[1].scatter(face_counts, mask_coverage * 100, alpha=0.5, color='coral', edgecolors='black', linewidth=0.5)
axes[1].set_title('Mask Coverage vs Number of Faces')
axes[1].set_xlabel('Number of Faces')
axes[1].set_ylabel('Coverage (%)')

plt.tight_layout()
plt.savefig('eda_mask_coverage.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nMask Coverage Stats:')
print(f'  Mean: {mask_coverage.mean()*100:.2f}%')
print(f'  Median: {np.median(mask_coverage)*100:.2f}%')
print(f'  Min: {mask_coverage.min()*100:.2f}%')
print(f'  Max: {mask_coverage.max()*100:.2f}%')

## 1.10 Pixel Intensity Analysis

In [ ]:
# Sample pixel intensity distributions
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
colors = ['red', 'green', 'blue']
channel_names = ['Red', 'Green', 'Blue']

# Aggregate pixel stats from a sample of images
sample_size = min(50, len(data))
sample_idx = np.random.choice(len(data), sample_size, replace=False)

for ch in range(3):
    all_pixels = []
    for idx in sample_idx:
        img = data[idx][0]
        if len(img.shape) == 2:
            img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
        elif img.shape[2] == 4:
            img = img[:, :, :3]
        all_pixels.extend(img[:, :, ch].flatten()[::10])  # subsample for speed
    
    axes[ch].hist(all_pixels, bins=50, color=colors[ch], alpha=0.7, edgecolor='black')
    axes[ch].set_title(f'{channel_names[ch]} Channel Distribution')
    axes[ch].set_xlabel('Pixel Value')
    axes[ch].set_ylabel('Frequency')

plt.suptitle('Pixel Intensity Distribution (sampled)', fontsize=14)
plt.tight_layout()
plt.savefig('eda_pixel_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 1.11 EDA Summary

### Key Findings:
- **409 images** with face bounding-box annotations
- **Variable image sizes** (180-4016px height, 182-6016px width) → need resizing to 256×256
- **Mixed channels**: Mostly RGB (3ch), with 1 grayscale and some RGBA → need standardization to 3 channels
- **1-16 faces per image**, majority have 1-3 faces
- **Annotations are bounding boxes** (normalized coordinates), not pixel-level masks → we generate binary masks
- **Class imbalance**: Face regions are small relative to image area (low mask coverage) → Dice Loss will help

### Next Steps (Task 2):
- Resize all images and masks to 256×256
- Normalize pixel values to [0, 1]
- Standardize all images to 3 channels (RGB)
- Train/validation split
- Data augmentation (flip, rotate, brightness)

In [ ]:
# Save processed data for Task 2
print('Saving processed data for next task...')
np.save('all_masks.npy', np.array(all_masks, dtype=object))
print('Saved: all_masks.npy')
print('\nTask 1 Complete! Ready for Task 2: Data Preprocessing & Augmentation')